# DeBERTa-v3-base PCL Detection (Trainer API, Augmented)

This notebook uses a simplified Hugging Face Trainer pipeline for Subtask 1 and keeps an explicit augmentation step on the minority (PCL) class.

Core stack:
- `AutoModelForSequenceClassification`
- `Trainer`
- `TrainingArguments`

Best checkpoint is selected by **PCL-class F1** (`f1_pcl`).


In [ ]:
!pip install -q huggingface_hub nlpaug sacremoses google-genai nltk contractions


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login

login(token=hf_token)


In [ ]:
import os
import re
import random
import inspect
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import wordnet, stopwords as nltk_stopwords

import nlpaug.augmenter.word as naw

try:
    from google import genai
    from google.genai import types as genai_types
    GENAI_AVAILABLE = True
except ImportError:
    GENAI_AVAILABLE = False
    print("WARNING: google-genai not installed. Gemini paraphrasing will be skipped.")

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

for _res in ['wordnet', 'stopwords', 'punkt', 'omw-1.4', 'averaged_perceptron_tagger']:
    nltk.download(_res, quiet=True)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# ============================================================
# Configuration
# ============================================================
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 256
NUM_LABELS = 2

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

# Augmentation controls
SYNONYM_N_MIN = 1            # min words replaced per sentence (WordNet)
SYNONYM_N_MAX = 3            # max words replaced per sentence (WordNet)
OVERSAMPLE_TARGET_PCL = 0.33 # target PCL fraction after all augmentation (~1:2)
BACK_TRANS_BATCH = 8         # texts per nlpaug call
BACK_TRANS_LIMIT = None      # None = all PCL; int = cap at N samples (for speed)
SKIP_BACK_TRANS = False      # set True to skip back translation
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
GEMINI_MODEL = 'gemini-2.5-flash'
GEMINI_BATCH_SIZE = 100      # samples per Gemini request (large context window)
GEMINI_TEMP = 0.8
SKIP_GEMINI = not (GENAI_AVAILABLE and bool(GEMINI_API_KEY))

EVAL_EVERY = 20
LOG_EVERY = 50

if os.path.exists('/kaggle/input'):
    DATA_ROOT = '/kaggle/input/datasets/wowthecoder/patronizing-and-condescending-language-detection'
else:
    DATA_ROOT = '..'

TSV_PATH = os.path.join(DATA_ROOT, 'dontpatronizeme_pcl.tsv')
TRAIN_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv')
DEV_IDS_PATH = os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv')

OUTPUT_DIR = 'checkpoints/deberta_v3_augmented_trainer'
LOG_DIR = 'logs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f'DATA_ROOT       : {DATA_ROOT}')
print(f'OUTPUT_DIR      : {OUTPUT_DIR}')
print(f'SKIP_BACK_TRANS : {SKIP_BACK_TRANS}')
print(f'SKIP_GEMINI     : {SKIP_GEMINI}')


In [ ]:
def load_task1(train_path: str) -> pd.DataFrame:
    """
    Load Task 1 data and convert original labels to binary:
      0/1 -> 0 (No-PCL)
      2/3/4 -> 1 (PCL)
    """
    rows = []
    with open(train_path, encoding='utf-8') as f:
        for line in f.readlines()[4:]:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 6:
                continue

            par_id = parts[0]
            art_id = parts[1]
            keyword = parts[2]
            country = parts[3]
            text = parts[4]
            orig_label = parts[-1]
            label = 0 if orig_label in {'0', '1'} else 1

            rows.append(
                {
                    'par_id': str(par_id),
                    'art_id': art_id,
                    'keyword': keyword,
                    'country': country,
                    'text': text,
                    'label': label,
                    'orig_label': orig_label,
                }
            )

    return pd.DataFrame(
        rows,
        columns=['par_id', 'art_id', 'keyword', 'country', 'text', 'label', 'orig_label'],
    )


def preprocess_text(text: str) -> str:
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df = load_task1(TSV_PATH)
df['clean_text'] = df['text'].apply(preprocess_text)

print(f'Loaded dataset: {len(df):,} samples')
print(df['label'].value_counts().sort_index().rename({0: 'No-PCL', 1: 'PCL'}))


In [ ]:
# ============================================================
# Official Train/Dev split
# ============================================================
train_ids_df = pd.read_csv(TRAIN_IDS_PATH, dtype={'par_id': str})
dev_ids_df = pd.read_csv(DEV_IDS_PATH, dtype={'par_id': str})

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids = set(dev_ids_df['par_id'].astype(str))

train_df = df[df['par_id'].isin(train_par_ids)].copy().reset_index(drop=True)
dev_df = df[df['par_id'].isin(dev_par_ids)].copy().reset_index(drop=True)

leftover_df = df[~df['par_id'].isin(train_par_ids | dev_par_ids)].copy().reset_index(drop=True)
if len(leftover_df) > 0:
    train_df = pd.concat([train_df, leftover_df], ignore_index=True)
    print(f'Appended {len(leftover_df):,} unassigned samples to training set.')


def describe_split(name: str, frame: pd.DataFrame):
    n = len(frame)
    n_pcl = int((frame['label'] == 1).sum())
    n_no_pcl = int((frame['label'] == 0).sum())
    ratio = f'1:{(n_no_pcl / n_pcl):.1f}' if n_pcl > 0 else 'undefined'
    print(f'{name:<8} -> total={n:,} | PCL={n_pcl:,} | No-PCL={n_no_pcl:,} | ratio={ratio}')


describe_split('Train(raw)', train_df)
describe_split('Dev(raw)', dev_df)


In [ ]:
# ============================================================
# Augmentation 1 — Synonym Replacement (WordNet)
# Replaces SYNONYM_N_MIN..SYNONYM_N_MAX non-stopword content
# words per sentence with a random WordNet synonym.
# Applied to all PCL training samples.
# ============================================================
STOP_WORDS = set(nltk_stopwords.words('english'))


def get_wordnet_synonyms(word):
    """Return WordNet synonyms for word (single-token, different from original)."""
    syns = set()
    for synset in wordnet.synsets(word):
        for lemma in synset.lemmas():
            candidate = lemma.name().replace('_', ' ').lower()
            if candidate != word.lower() and ' ' not in candidate:
                syns.add(candidate)
    return list(syns)


def synonym_replacement(text, n_min=SYNONYM_N_MIN, n_max=SYNONYM_N_MAX):
    """Replace n random content words with WordNet synonyms (1-3 per sentence)."""
    words = text.split()
    n = random.randint(n_min, min(n_max, len(words)))
    candidates = [
        i for i, w in enumerate(words)
        if w.isalpha() and w.lower() not in STOP_WORDS
    ]
    random.shuffle(candidates)
    new_words = words.copy()
    replaced = 0
    for idx in candidates:
        if replaced >= n:
            break
        syns = get_wordnet_synonyms(words[idx])
        if syns:
            new_words[idx] = random.choice(syns)
            replaced += 1
    return ' '.join(new_words)


pcl_train = train_df[train_df['label'] == 1].copy().reset_index(drop=True)

syn_rows = []
for _, row in pcl_train.iterrows():
    aug_text = synonym_replacement(row['clean_text'])
    if aug_text and aug_text != row['clean_text']:
        new_row = row.copy()
        new_row['par_id'] = str(row['par_id']) + '_syn'
        new_row['clean_text'] = aug_text
        syn_rows.append(new_row)

aug_syn_df = pd.DataFrame(syn_rows).reset_index(drop=True)
print(f'Synonym replacement: {len(aug_syn_df)} new PCL samples generated.')


In [ ]:
# ============================================================
# Augmentation 2 — Back Translation (nlpaug + FairSeq WMT19)
# Translates PCL sentences EN -> DE -> EN using Facebook's
# WMT19 FairSeq models via HuggingFace. Set SKIP_BACK_TRANS=True
# to skip (saves ~2GB download and significant compute time).
# ============================================================
if SKIP_BACK_TRANS:
    aug_bt_df = pd.DataFrame(columns=train_df.columns)
    print('Back translation SKIPPED (SKIP_BACK_TRANS=True).')
else:
    print('Loading FairSeq WMT19 models (facebook/wmt19-en-de, facebook/wmt19-de-en). '
          'First run downloads ~2GB ...')
    try:
        bt_aug = naw.BackTranslationAug(
            from_model_name='facebook/wmt19-en-de',
            to_model_name='facebook/wmt19-de-en',
            device='cuda' if torch.cuda.is_available() else 'cpu',
            max_length=512,
            batch_size=BACK_TRANS_BATCH,
        )
        print('Back-translation augmenter ready.')

        pcl_bt_pool = pcl_train if BACK_TRANS_LIMIT is None else pcl_train.head(BACK_TRANS_LIMIT)
        texts = pcl_bt_pool['clean_text'].tolist()
        bt_texts = []

        for i in range(0, len(texts), BACK_TRANS_BATCH):
            batch = texts[i : i + BACK_TRANS_BATCH]
            try:
                augmented = bt_aug.augment(batch)
                # nlpaug may return list-of-lists depending on version
                if augmented and isinstance(augmented[0], list):
                    augmented = [a[0] if a else b for a, b in zip(augmented, batch)]
                bt_texts.extend(augmented)
            except Exception as e:
                print(f'  Batch {i // BACK_TRANS_BATCH} failed: {e} — keeping originals')
                bt_texts.extend(batch)
            if (i // BACK_TRANS_BATCH + 1) % 10 == 0:
                print(f'  ... {min(i + BACK_TRANS_BATCH, len(texts))}/{len(texts)} done')

        bt_rows = []
        for (_, row), bt_text in zip(pcl_bt_pool.iterrows(), bt_texts):
            # Re-apply preprocessing to translated output
            cleaned = re.sub(r'[^a-zA-Z\s]', ' ', str(bt_text))
            cleaned = re.sub(r'\s+', ' ', cleaned).strip()
            if cleaned and cleaned != row['clean_text']:
                new_row = row.copy()
                new_row['par_id'] = str(row['par_id']) + '_bt'
                new_row['clean_text'] = cleaned
                bt_rows.append(new_row)

        aug_bt_df = pd.DataFrame(bt_rows).reset_index(drop=True)
        print(f'Back translation: {len(aug_bt_df)} new PCL samples generated.')

    except Exception as e:
        print(f'Back translation setup failed: {e}')
        print('Continuing without back translation.')
        aug_bt_df = pd.DataFrame(columns=train_df.columns)


In [ ]:
# ============================================================
# Augmentation 3 — Gemini Paraphrasing (Gemini 2.5 Flash)
# Sends GEMINI_BATCH_SIZE PCL samples per request, exploiting
# Gemini's large context window to paraphrase efficiently.
# The system prompt ensures patronising tone is preserved.
# Requires: export GEMINI_API_KEY=<your_key>
# ============================================================
GEMINI_SYSTEM_PROMPT = """You are an expert linguist and data scientist specializing in identifying and generating Patronizing and Condescending Language (PCL).

Your task is to paraphrase sentences to augment a machine learning dataset. You MUST strictly adhere to these rules:
1. PRESERVE THE TONE: You must maintain the exact same patronizing tone, "savior complex", pity, or implicit superiority found in the original text if the label indicates that the text contains PCL.
2. DO NOT SANITIZE: Do not neutralize the text into polite, objective English.
3. NO OVERT TOXICITY: PCL is subtle. Do not use slurs, aggressive hate speech, or overt hostility if not present in the original text.
4. NO CHATTER: Output ONLY the paraphrased sentence. Do not include quotes or explanations."""


def parse_gemini_response(response_text, expected_count):
    """Extract numbered list items from Gemini response."""
    paraphrases = {}
    for line in response_text.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        m = re.match(r'^(\d+)[.):]\s*(.+)$', line)
        if m:
            idx = int(m.group(1))
            text = m.group(2).strip().strip('\"\' ')
            if 1 <= idx <= expected_count and text:
                paraphrases[idx] = text
    return paraphrases


def gemini_paraphrase_batch(client, texts):
    """Paraphrase a batch of texts. Returns list of same length (falls back to originals on error)."""
    n = len(texts)
    numbered = '\n'.join(f'{i+1}. {t}' for i, t in enumerate(texts))
    user_prompt = (
        f'Paraphrase each of the following {n} sentences. '
        f'Output ONLY a numbered list in the exact same format (number. text), '
        f'one paraphrase per line, no extra commentary:\n\n{numbered}'
    )
    try:
        response = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=user_prompt,
            config=genai_types.GenerateContentConfig(
                system_instruction=GEMINI_SYSTEM_PROMPT,
                temperature=GEMINI_TEMP,
            ),
        )
        parsed = parse_gemini_response(response.text, n)
        return [parsed.get(i + 1, texts[i]) for i in range(n)]
    except Exception as e:
        print(f'  Gemini batch failed: {e}')
        return texts


if SKIP_GEMINI:
    aug_gemini_df = pd.DataFrame(columns=train_df.columns)
    print('Gemini paraphrasing SKIPPED (set GEMINI_API_KEY env var to enable).')
else:
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    pcl_texts = pcl_train['clean_text'].tolist()
    print(f'Paraphrasing {len(pcl_texts)} PCL samples with {GEMINI_MODEL} '
          f'({GEMINI_BATCH_SIZE} per request) ...')

    all_paraphrases = []
    for start in range(0, len(pcl_texts), GEMINI_BATCH_SIZE):
        batch_texts = pcl_texts[start : start + GEMINI_BATCH_SIZE]
        batch_paras = gemini_paraphrase_batch(gemini_client, batch_texts)
        all_paraphrases.extend(batch_paras)
        done = min(start + GEMINI_BATCH_SIZE, len(pcl_texts))
        print(f'  ... {done}/{len(pcl_texts)} paraphrased')
        time.sleep(1)  # brief pause to respect rate limits

    gemini_rows = []
    for row, para in zip(pcl_train.itertuples(index=False), all_paraphrases):
        cleaned = re.sub(r'[^a-zA-Z\s]', ' ', str(para))
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()
        if cleaned and cleaned != row.clean_text:
            gemini_rows.append({
                'par_id':     str(row.par_id) + '_gemini',
                'art_id':     row.art_id,
                'keyword':    row.keyword,
                'country':    row.country,
                'text':       row.text,
                'clean_text': cleaned,
                'label':      1,
                'orig_label': row.orig_label,
            })
    aug_gemini_df = pd.DataFrame(gemini_rows).reset_index(drop=True)
    print(f'Gemini paraphrasing: {len(aug_gemini_df)} new PCL samples generated.')


In [ ]:
# ============================================================
# Augmentation 4 — Combine + Random Oversampling
# Merges all augmented DataFrames, then randomly duplicates
# PCL samples until OVERSAMPLE_TARGET_PCL fraction is reached.
# ============================================================

def random_oversample_to_target(df_in, target_pcl_fraction, seed=SEED):
    """Randomly duplicate PCL samples to reach target_pcl_fraction of total."""
    pcl    = df_in[df_in['label'] == 1]
    no_pcl = df_in[df_in['label'] == 0]
    if len(pcl) == 0:
        return df_in.copy()
    target_pcl_count = int(len(no_pcl) * target_pcl_fraction / (1.0 - target_pcl_fraction))
    if target_pcl_count <= len(pcl):
        print(f'Oversampling not needed (PCL {len(pcl)} >= target {target_pcl_count}).')
        return df_in.copy()
    extra_needed = target_pcl_count - len(pcl)
    extra = pcl.sample(n=extra_needed, replace=True, random_state=seed).copy()
    extra['par_id'] = [f'{pid}_os{i}' for i, pid in enumerate(extra['par_id'])]
    out = pd.concat([df_in, extra], ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    print(f'Oversampling: added {extra_needed} duplicate PCL samples '
          f'(PCL: {len(pcl)} -> {target_pcl_count}, target fraction: {target_pcl_fraction:.0%})')
    return out


# Combine original train + all augmented frames
train_aug_df = pd.concat(
    [train_df, aug_syn_df, aug_bt_df, aug_gemini_df],
    ignore_index=True,
)
train_aug_df['label'] = train_aug_df['label'].astype(int)

# Random oversampling to top up to target ratio
train_final_df = random_oversample_to_target(train_aug_df, OVERSAMPLE_TARGET_PCL, seed=SEED)


def describe(name, frame):
    n = len(frame)
    n_pcl = int((frame['label'] == 1).sum())
    n_no  = int((frame['label'] == 0).sum())
    frac  = (n_pcl / n) if n > 0 else 0.0
    print(f'{name:<14} -> total={n:,} | PCL={n_pcl:,} ({frac:.1%}) | No-PCL={n_no:,}')


describe('Train(raw)',   train_df)
describe('Train(syn)',   pd.concat([train_df, aug_syn_df]))
describe('Train(+BT)',   pd.concat([train_df, aug_syn_df, aug_bt_df]))
describe('Train(+gem)',  train_aug_df)
describe('Train(final)', train_final_df)
describe('Dev',          dev_df)


In [ ]:
# ============================================================
# Tokenization + Hugging Face Datasets
# ============================================================
train_hf = Dataset.from_pandas(
    train_final_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}),
    preserve_index=False,
)
dev_hf = Dataset.from_pandas(
    dev_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}),
    preserve_index=False,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)


train_ds = train_hf.map(tokenize, batched=True, remove_columns=['text'])
dev_ds = dev_hf.map(tokenize, batched=True, remove_columns=['text'])

train_ds = train_ds.rename_column('label', 'labels')
dev_ds = dev_ds.rename_column('label', 'labels')

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
dev_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print(dev_ds)


In [ ]:
# ============================================================
# Metrics
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_no_pcl': f1_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'f1_pcl': f1_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'prec_no_pcl': precision_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'prec_pcl': precision_score(labels, preds, pos_label=1, average='binary', zero_division=0),
        'rec_no_pcl': recall_score(labels, preds, pos_label=0, average='binary', zero_division=0),
        'rec_pcl': recall_score(labels, preds, pos_label=1, average='binary', zero_division=0),
    }


In [ ]:
# ============================================================
# Model + TrainingArguments + Trainer
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

args_kwargs = {
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': NUM_EPOCHS,
    'per_device_train_batch_size': TRAIN_BATCH_SIZE,
    'per_device_eval_batch_size': EVAL_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'warmup_ratio': WARMUP_RATIO,
    'lr_scheduler_type': 'cosine',
    'logging_steps': LOG_EVERY,
    'save_strategy': 'steps',
    'save_steps': EVAL_EVERY,
    'eval_steps': EVAL_EVERY,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1_pcl',
    'greater_is_better': True,
    'save_total_limit': 2,
    'report_to': 'none',
    'seed': SEED,
    'fp16': torch.cuda.is_available(),
}

# Handle transformers API change: evaluation_strategy -> eval_strategy
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in ta_sig:
    args_kwargs['eval_strategy'] = 'steps'
else:
    args_kwargs['evaluation_strategy'] = 'steps'

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer configured.')


In [ ]:
train_result = trainer.train()
train_result


In [ ]:
eval_metrics = trainer.evaluate()
print('Evaluation metrics:')
for k, v in eval_metrics.items():
    if isinstance(v, (int, float)):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')


In [ ]:
# ============================================================
# Detailed evaluation report
# ============================================================
pred_output = trainer.predict(dev_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

print(classification_report(y_true, y_pred, target_names=['No-PCL', 'PCL'], digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Pred No-PCL', 'Pred PCL'],
    yticklabels=['True No-PCL', 'True PCL'],
)
plt.title('Confusion Matrix (Dev)')
plt.tight_layout()
plt.show()


In [ ]:
BEST_MODEL_DIR = os.path.join(OUTPUT_DIR, 'best')
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f'Saved best model and tokenizer to: {BEST_MODEL_DIR}')
